# Scientific ToT retrieval with PyTerrier

This notebook uses the canonical adapter, Terrier's BM25, and `pyterrier.Experiment` with `ir_measures` metrics. It retrieves over the full available document collection.

From the repository root:
```sh
uv sync --locked
uv run python -m ipykernel install --sys-prefix --name scientific-tot --display-name "Scientific ToT"
uv run jupyter lab
```
Select **Scientific ToT** for both notebooks. The default input is the restored data at `../multimodal-tip-of-tongue-for-scientific-papers/data`. Set `TOT_DATA_ROOT` if your data lives elsewhere. By default, all available documents in the selected domain and all test queries are used with fulltext. For the two-document smoke test, set `TOT_DATA_ROOT=tests/fixtures/adapter` when launching from the repository root. The default output is a fresh directory per configuration-cell execution; an explicit output environment variable must name a fresh directory. The first BM25 run downloads Terrier's JAR; the first dense run downloads model weights. Java is supplied by the locked environment automatically.


In [ ]:
import os
import sys
from pathlib import Path
from uuid import uuid4

from scientific_tot import Dataset, DatasetAdapter
from scientific_tot.integrations.pyterrier import bm25_experiment, to_pyterrier
from scientific_tot.integrations.selection import load_experiment_data

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Select the Scientific ToT kernel registered from this project's .venv")
DATA = Path(
    os.environ.get(
        "TOT_DATA_ROOT", str(ROOT.parent / "multimodal-tip-of-tongue-for-scientific-papers/data")
    )
)


DATA = DATA.expanduser()
if not DATA.is_absolute():
    DATA = ROOT / DATA

OUTPUT = Path(
    os.environ.get("TOT_OUTPUT_DIR", str(ROOT / "results" / f"bm25-notebook-{uuid4().hex[:8]}"))
)
DOMAIN = "ACL"
DOCUMENTS, QUERIES = None, None  # All documents/test queries; use 250, 10 for a demo.
CORPUS_SPLIT = "all"  # Include every available extraction, including distractors.
MISSING = "error"  # "skip" explicitly excludes incomplete records.
print("Python:", sys.executable, "Data:", DATA, "Output:", OUTPUT)

In [ ]:
adapter = DatasetAdapter(Dataset(local_dir=DATA))
corpora, queries = load_experiment_data(
    adapter,
    domain=DOMAIN,
    corpus_split=CORPUS_SPLIT,
    missing=MISSING,
    documents=DOCUMENTS,
    query_limit=QUERIES,
)
inputs = to_pyterrier(next(iter(corpora.values())), queries)
print(len(inputs.documents), "documents;", len(inputs.topics), "queries")
inputs.topics.head()

Retrieval indexes the full original document text. Bibliographic metadata is not required. Set `MISSING = "skip"` only to explicitly exclude missing fulltext; all relevance targets must still resolve.


In [ ]:
corpus = corpora["fulltext"]
print(len(corpus), "documents", corpus.report.to_dict())

In [ ]:
metrics = bm25_experiment(corpus, queries, OUTPUT / "fulltext")
metrics

Each representation directory contains `run.trec`, `metrics.csv`, `per-query.csv`, `topics.jsonl`, `qrels.tsv`, a reversible `query-ids.jsonl` map, and `manifest.json`. `RR@10` is MRR@10, `R@1` is Recall@1, `R@10` is Recall@10, and `nDCG@10` is NDCG@10. All metrics come from standard library implementations.

The default uses every available document in the domain and all selected test queries. If limits are set explicitly, selection includes all positive targets, then fills remaining places using sorted canonical IDs.

In [ ]:
import pyterrier as pt

run = pt.io.read_results(str(OUTPUT / "fulltext" / "run.trec"))
assert set(run.docno).issubset({r["doc_id"] for r in corpus})
run[["qid", "docno", "rank", "score"]].head(10)